# Causal Inference in Practice
## Week 5 — Regression & Adjustment · Practice Notebook

> **Block II — Adjustment for confounding**
>
> What regression can and cannot do for causation — and the precise ways that "controlling for" backfires.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · Regression as covariate adjustment

We start where Week 1 left off: a confounder `Z` pushes both the treatment `X` and the outcome `Y`. The **true** total effect of `X` on `Y` is exactly `2`. A naive regression credits part of `Z`'s effect to `X`; adjusting for `Z` recovers the truth. The point of this notebook is to see — on data where we *know* the answer — exactly when an OLS slope is a causal effect and the precise ways it stops being one.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

TRUE_EFFECT = 2.0          # the ground truth we will keep checking
n = 8000
Z = RNG.normal(size=n)                          # confounder Z -> X, Z -> Y
X = 0.9 * Z + RNG.normal(size=n)                # treatment
Y = TRUE_EFFECT * X + 1.5 * Z + RNG.normal(size=n)   # true X->Y = 2

naive    = sm.OLS(Y, sm.add_constant(X)).fit().params[1]
adjusted = sm.OLS(Y, sm.add_constant(np.c_[X, Z])).fit().params[1]
print(f'naive    = {naive:.3f}   (biased: Z leaks into X)')
print(f'adjusted = {adjusted:.3f}   vs truth {TRUE_EFFECT}')
assert abs(adjusted - TRUE_EFFECT) < 0.15, 'adjusting for Z should recover 2'
assert naive - adjusted > 0.3, 'naive should be visibly biased upward'

**Frisch–Waugh–Lovell:** 'controlling for `Z`' literally means partialling `Z` out of *both* `X` and `Y`, then relating the residuals. Let's confirm the adjusted slope equals the residual-on-residual slope — regression really is a matching/comparison estimator in disguise.

In [ ]:
def resid(target, *regs):
    """Residual of `target` after regressing it on the given regressors."""
    A = sm.add_constant(np.column_stack(regs))
    fit = sm.OLS(target, A).fit()
    return target - fit.predict(A)

rX = resid(X, Z)        # X with Z partialled out
rY = resid(Y, Z)        # Y with Z partialled out
fwl = sm.OLS(rY, rX).fit().params[0]    # no intercept needed on residuals
print(f'residual-on-residual slope = {fwl:.3f}')
print(f'multivariate adjusted slope = {adjusted:.3f}')
assert abs(fwl - adjusted) < 1e-6, 'FWL: the two slopes must match exactly'
print('FWL holds: controlling for Z == partialling Z out of both sides.')

## 2 · Good controls vs. bad controls

Same machinery, four roles. We extend the system with a **mediator** `M` (on the path `X → M → Y`), a **collider** `K` (a common effect, `X → K ← Y`), and a **neutral** predictor `V` — an independent cause of `Y` (`V → Y`) that has no relationship with `X`. Watch what adjusting for each does to the estimated `X → Y` effect. The truth is still `2`.

In [ ]:
# Construction: a direct effect of 1.0 plus an indirect effect of 1.0
# through the mediator M, for a TOTAL effect of X on Y of 2.0 — our truth.
M    = 1.0 * X + RNG.normal(size=n)            # X -> M  (path coeff 1.0)
V    = RNG.normal(size=n)                      # neutral: an independent cause of Y
Yc   = 1.0 * X + 1.0 * M + 1.5 * Z + 1.5 * V + RNG.normal(size=n)  # direct 1.0 + M*1.0
K    = 1.0 * X + 1.0 * Yc + RNG.normal(size=n) # collider X -> K <- Y

def xslope(y, *cols):
    """OLS slope on X (first regressor), adjusting for the rest."""
    return sm.OLS(y, sm.add_constant(np.column_stack((X,) + cols))).fit().params[1]

good     = xslope(Yc, Z)          # adjust confounder only  -> total effect 2
with_med = xslope(Yc, Z, M)       # + mediator (bad)        -> blocks indirect path
with_col = xslope(Yc, Z, K)       # + collider (bad)        -> opens fake path
with_neu = xslope(Yc, Z, V)       # + neutral predictor     -> harmless
print(f'adjust Z only       = {good:.3f}   <- truth 2.0')
print(f'+ mediator M        = {with_med:.3f}   (drops ~1.0: the indirect path is blocked)')
print(f'+ collider K        = {with_col:.3f}   (biased: a fake path opened)')
print(f'+ neutral V         = {with_neu:.3f}   (still ~2.0: harmless)')

Read the four numbers. **Only the good and the neutral controls left the estimate at the truth.** The mediator removed the part of the effect that runs through it; the collider opened a spurious path. The neutral predictor `V` didn't move the point estimate — and because it explains outcome variance, it actually *tightens* it. That precision payoff is the whole reason to keep a neutral control.

In [ ]:
# The neutral control is harmless for bias AND helps precision:
se_without = sm.OLS(Yc, sm.add_constant(np.c_[X, Z])).fit().bse[1]
se_with    = sm.OLS(Yc, sm.add_constant(np.c_[X, Z, V])).fit().bse[1]
print(f'SE on X  without V = {se_without:.4f}   with V = {se_with:.4f}')
assert se_with < se_without, 'a neutral outcome-predictor should shrink the SE on X'

# Make the bias lesson machine-checkable against the known truth.
assert abs(good - 2.0) < 0.15,        'confounder adjustment should recover 2'
assert abs(with_neu - 2.0) < 0.15,    'neutral control should not bias the estimate'
assert abs(with_med - 2.0) > 0.5,     'mediator adjustment SHOULD bias it (away from 2)'
assert abs(with_col - 2.0) > 0.3,     'collider adjustment SHOULD bias it (away from 2)'
print('All four roles behaved as the DAG predicts. Over-adjustment is not caution.')

### 🔧 Exercise 2.1 — the birth-weight paradox (a collider story)

Reconstruct the famous reversal. Maternal **smoking** `S` lowers **birth weight** `BW`. An unmeasured factor `U` (think birth defects) *also* lowers birth weight **and** independently raises infant **mortality** `Mort`. Crucially, in this simulation smoking has **no real effect on mortality** — the true coefficient is `0`.

Show that (a) the unadjusted smoking–mortality association is ~0 (the truth), but (b) **stratifying on / adjusting for birth weight — a collider — makes smoking look protective** (a negative coefficient).

Fill in the `# TODO`s.

In [ ]:
# TODO: build S, U, BW (collider of S and U), and Mort (caused by U, NOT by S).
# true effect of smoking on mortality is 0.0
S    = RNG.binomial(1, 0.4, size=n)              # smoking (0/1)
U    = RNG.normal(size=n)                        # unmeasured (e.g. defects)
BW   = ...        # TODO: lower with smoking AND with U  (e.g. 3300 - 200*S - 300*U + noise)
Mort = ...        # TODO: rises with U only, NOT with S  (e.g. 0.5*U + noise)
# unadj = ...     # TODO: slope of Mort on S, no adjustment   -> ~0
# adj   = ...     # TODO: slope of Mort on S, adjusting for BW -> negative (the 'paradox')
# print(unadj, adj)

### ✅ Solution 2.1

In [ ]:
S    = RNG.binomial(1, 0.4, size=n).astype(float)   # smoking
U    = RNG.normal(size=n)                               # unmeasured cause
BW   = 3300 - 200*S - 300*U + RNG.normal(0, 100, size=n) # collider: S and U both lower it
Mort = 0.5*U + RNG.normal(0, 0.5, size=n)               # U raises mortality; S does NOT

unadj = sm.OLS(Mort, sm.add_constant(S)).fit().params[1]            # crude S->Mort
adj   = sm.OLS(Mort, sm.add_constant(np.c_[S, BW])).fit().params[1] # adjusting the collider BW
print(f'smoking->mortality, unadjusted     = {unadj:+.4f}   (truth 0)')
print(f'smoking->mortality, adjusting BW    = {adj:+.4f}   (now looks PROTECTIVE!)')
assert abs(unadj) < 0.10,  'with no real effect, the crude association is ~0'
assert adj < -0.15,        'adjusting for the collider BW invents a protective effect'
print('\nBirth weight is a COLLIDER of smoking and U. Conditioning on it')
print('opens S -> BW <- U -> Mort and manufactures a spurious benefit.')

## 3 · The Table 2 fallacy

A single regression hands you a coefficient for **every** regressor. The Table 2 fallacy is reading each one as that variable's causal effect. We fit the *correct* model for `X` — `Y ~ X + Z` — and show that the coefficient on the confounder `Z` is **not** `Z`'s own total causal effect, even though the coefficient on `X` is right.

In [ ]:
# A system where we know BOTH true effects:
#   true total effect of X on Y = 2.0
#   Z affects Y directly (1.5) AND through X (0.9 * 2.0) -> Z's TOTAL effect = 1.5 + 1.8 = 3.3
Zt = RNG.normal(size=n)
Xt = 0.9 * Zt + RNG.normal(size=n)
Yt = 2.0 * Xt + 1.5 * Zt + RNG.normal(size=n)

fit = sm.OLS(Yt, sm.add_constant(np.c_[Xt, Zt])).fit()
coef_X = fit.params[1]            # this IS the causal effect of X (Z is its confounder)
coef_Z = fit.params[2]            # this is NOT Z's causal effect
true_total_Z = sm.OLS(Yt, sm.add_constant(Zt)).fit().params[1]   # Z's actual total effect

print(f'coef on X in the model      = {coef_X:.3f}   (= true effect of X, 2.0)')
print(f'coef on Z in the SAME model = {coef_Z:.3f}   (the DIRECT effect, ~1.5)')
print(f"Z's true TOTAL effect       = {true_total_Z:.3f}   (~3.3)")
assert abs(coef_X - 2.0) < 0.15
assert abs(coef_Z - 1.5) < 0.2          # the in-model Z coef is the DIRECT effect
assert abs(true_total_Z - 3.3) < 0.2    # Z's true total effect is bigger
assert abs(coef_Z - true_total_Z) > 1.0 # ... and they are NOT the same number

Same regression, two coefficients — and only the one we **designed the model for** is a clean causal effect. The `Z` coefficient is `Z`'s effect *holding `X` fixed*, i.e. the direct effect, because `X` is a **mediator** on `Z`'s path to `Y`. To get `Z`'s total effect you would build a *different* model. That is the Table 2 fallacy: **one model, one trustworthy coefficient.**

### 🔧 Exercise 3.1 — give Z its own (correct) model

Using the `Xt, Zt, Yt` from the previous cell, estimate `Z`'s **total** causal effect *correctly*. Think about `Z`'s back-door paths: is `X` a confounder of `Z → Y`, or a mediator? Decide what (if anything) to adjust for, then recover ~`3.3`.

Fill in the `# TODO`.

In [ ]:
# TODO: build the right model for the TOTAL effect of Z on Y.
# Hint: X lies on a path Z -> X -> Y, so X is a MEDIATOR for Z's total effect.
# total_Z = ...   # TODO: regress Y on Z WITHOUT adjusting for the mediator X
# print(total_Z)

### ✅ Solution 3.1

In [ ]:
# X is a mediator of Z's effect on Y, so we must NOT adjust for it.
total_Z = sm.OLS(Yt, sm.add_constant(Zt)).fit().params[1]
print(f"Z's total effect (X left out, as a mediator) = {total_Z:.3f}   (truth ~3.3)")
assert abs(total_Z - 3.3) < 0.2, 'leaving the mediator X out recovers Z total effect'
print('Each variable needs its OWN adjustment set: X for X, none-but-Z for Z.')

## 4 · Interaction & effect modification

A single slope assumes one effect for everyone. Here the treatment effect genuinely **differs by subgroup** `G`: it is `1.0` when `G=0` and `3.0` when `G=1`. A model with `X` and `G` but no interaction reports an average that fits neither group; adding an `X×G` interaction recovers both conditional effects.

In [ ]:
G  = RNG.binomial(1, 0.5, size=n)                   # subgroup indicator
Xi = RNG.normal(size=n)                              # treatment (randomized here)
EFFECT = 1.0 + 2.0 * G                               # 1.0 if G=0, 3.0 if G=1
Yi = EFFECT * Xi + 0.5 * G + RNG.normal(size=n)
dfi = pd.DataFrame({'Y': Yi, 'X': Xi, 'G': G})

no_int = smf.ols('Y ~ X + G', data=dfi).fit().params['X']
print(f'single-slope model: effect of X = {no_int:.3f}  (an average that fits no one)')
mi = smf.ols('Y ~ X * G', data=dfi).fit()
eff_G0 = mi.params['X']                              # effect at G=0
eff_G1 = mi.params['X'] + mi.params['X:G']           # effect at G=1
print(f'interaction model:  effect at G=0 = {eff_G0:.3f}  (truth 1.0)')
print(f'interaction model:  effect at G=1 = {eff_G1:.3f}  (truth 3.0)')
assert abs(eff_G0 - 1.0) < 0.15, 'interaction should recover the G=0 effect'
assert abs(eff_G1 - 3.0) < 0.15, 'interaction should recover the G=1 effect'

The single-slope estimate (~2.0) is the average of the two true effects but describes neither subgroup. The interaction term lets the effect of `X` depend on `G` and recovers `1.0` and `3.0` on the nose. **Effect modification** (`G` changes the *size* of the effect) is a different phenomenon from **confounding** (`G` would *bias* the effect) — here `G` is independent of `X`, so it modifies without confounding.

### 🔧 Exercise 4.1 — recover the average treatment effect

From the interaction fit, the population **ATE** is the average of the subgroup effects weighted by subgroup size. With `G` split 50/50 it should be ~`2.0`. Compute it from the interaction coefficients and the observed mean of `G`, and check it matches the simple single-slope model's estimate.

Fill in the `# TODO`.

In [ ]:
# TODO: ATE = E[ effect(G) ] = effect_at_G0 + coef(X:G) * E[G]
# pG  = ...       # TODO: observed P(G=1)
# ate = ...       # TODO: combine eff_G0 and the interaction coefficient
# print(ate)

### ✅ Solution 4.1

In [ ]:
pG  = dfi['G'].mean()                               # P(G=1)
ate = eff_G0 + mi.params['X:G'] * pG                # weighted-average effect
print(f'ATE from interaction model = {ate:.3f}   (truth ~2.0)')
print(f'single-slope estimate      = {no_int:.3f}   (should be close)')
assert abs(ate - 2.0) < 0.15, 'weighted subgroup effects give the ATE'
assert abs(ate - no_int) < 0.2, 'and it matches the simple single-slope model'
print('The single slope was the ATE all along — it just hid the heterogeneity.')

## 5 · A picture: bias by what you adjust for

One figure to fix the intuition. We plot the estimated `X → Y` effect under four adjustment choices against the known truth. Good and neutral controls sit on the line; the mediator and collider do not.

In [ ]:
labels = ['adjust Z\n(good)', '+ mediator M\n(bad)',
          '+ collider K\n(bad)', '+ neutral V\n(ok)']
vals   = [good, with_med, with_col, with_neu]
colors = ['#2A9D8F', '#C0504D', '#C0504D', '#2F6DB5']

fig, ax = plt.subplots()
ax.bar(labels, vals, color=colors)
ax.axhline(2.0, color='#E9A23B', linestyle='--', linewidth=2, label='truth = 2.0')
ax.set_ylabel('estimated effect of X on Y')
ax.set_title('Only the good (and neutral) controls land on the truth')
ax.legend()
fig.tight_layout()
print('Figure built. Bars off the dashed line = bias you created by over-adjusting.')

## Wrap-up & self-check

- **Regression is adjustment.** Under the conditional independence assumption, the OLS slope on `X` is the causal effect; FWL shows it is residual-on-residual matching.
- **Classify every control.** Confounder → adjust; mediator → don't (for the total effect); collider → don't; neutral → optional (precision).
- **Over-adjustment is not caution.** You watched the mediator and collider move the estimate *away* from the known truth, and rebuilt the birth-weight paradox from a collider.
- **The Table 2 fallacy.** A regression gives one coefficient per regressor, but only the one you designed the model for is a clean causal effect; `Z` needs its own adjustment set.
- **Interactions** recover effects that differ by subgroup, and the weighted average returns the ATE.

**You're ready for Week 6** if you can classify each covariate on a DAG and say why a coefficient is not an effect. Next week: matching and propensity scores — adjusting by rebuilding a comparable control group instead of modeling the outcome.